# LongFlow P1 -- Capture v2 (dual-stream + sigma-bucket, long-form corpus)

Runtime: **L4 GPU**. Resumable: cache on Drive, safe to interrupt any time;
re-run cells 1-2 to continue. Target **~480K frame pairs** (matching v1's
478,191) via **~200-250 turn-split long-form renders spanning ~1-15 min**,
replacing v1's short LibriTTS clips -- directly targets the context-length
OOD failure axis GN1-8 diagnosed as dominant. Schema: hidden state for
**both cond and neg streams** (GN8 dual-stream fix) + DDPM target +
per-window sigma bucket (critic pass 2026-08-14 + GO note 2026-08-15).
Drag **`longflow_bundle.zip`** in when the cold start asks.

Pre-registered: `experiments/p1_flow_head/NOTES.md`
("2026-08-16 -- CAPTURE V2 INFRA BUILT"). Writes to a **new** Drive dir
(`longflow_p1_cache_v2`), separate from v1's cache -- do not mix schemas.
Mandatory 1K/5K gate check (hard constraint 6) before this cache feeds any
full training run.

In [ ]:
# ===== COLD START (idempotent) -- run me first, wait for READY =====
NOTEBOOK_VERSION = "Capture v2 v1.0 (2026-08-16): dual-stream + sigma-bucket, long-form corpus"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache_v2"  # NEW dir -- do not mix with v1's cache
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
os.makedirs(CACHE_DIR, exist_ok=True)

assert os.path.exists("/content/longflow_bundle.zip"), "DRAG longflow_bundle.zip IN, then re-run"
!cd /content && unzip -q -o longflow_bundle.zip
sys.path.insert(0, "/content")
from src.cache.capture import BatchedSampleCapture, UtteranceCache, save_utterance
from src.cache.noise import NoiseIntervention
FRAME_ID = processor.tokenizer.convert_tokens_to_ids("<|vision_pad|>")
print("READY")

In [ ]:
# ===== Corpus + noise-augmented batched capture loop (resumable) =====
import random
import uuid
from pathlib import Path
from datasets import load_dataset

TARGET_FRAMES = 480_000  # ~= v1's 478,191 frame pairs -- same total training-data budget,
                          # redistributed into long-form utterances instead of short clips
                          # (Josh's call, 2026-08-16 -- see NOTES.md)
WORD_BINS = [150, 300, 600, 1200, 2400]  # ~1/2/4/8/15 min @ ~165-190 wpm (N8 natural rate) --
                                          # spans short -> the multi-minute OOD range GN1-8 flagged
BIN_BATCH = {150: 8, 300: 8, 600: 6, 1200: 4, 2400: 2}  # smaller batches for longer bins (VRAM/wall-clock)
TURN_WORDS = 60  # N8 cure: same-speaker turn-splitting within one generate() call
WINDOW = 450  # ~60s @ 7.5Hz -- one sigma draw per window (GN6 ramp-schedule granularity)

def sigma_draw():
    r = random.random()
    if r < 0.5:
        return 0.0
    if r < 0.9:
        return round(random.uniform(0.1, 0.3), 2)
    return 0.4  # occasional, per the 2026-08-15 capture v2 GO note

def make_sigma_fn():
    cache = {}
    def sigma_fn(c):
        w = c // WINDOW
        if w not in cache:
            cache[w] = sigma_draw()
        return cache[w]
    return sigma_fn

def turnscript(sentences, target_words, turn_words=TURN_WORDS, speaker=1):
    """N8 cure: same-speaker turn-splitting into ~60-word turns within one
    generate() call, stopping once target_words total is reached."""
    turns, cur, w, total = [], [], 0, 0
    for s in sentences:
        cur.append(s)
        w += len(s.split())
        total += len(s.split())
        if w >= turn_words:
            turns.append(f"Speaker {speaker}: " + " ".join(cur))
            cur, w = [], 0
        if total >= target_words:
            break
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n", total

def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} -- retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern} -- check Drive mount/paths")

prompts = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")
ds = load_dataset("mythicinfinity/libritts_r", "clean", split="train.clean.360", streaming=True)

existing = list(Path(CACHE_DIR).glob("*.pt"))
done = {f.stem for f in existing}
frames_done = sum(torch.load(f, weights_only=True)["hidden"].shape[0] for f in existing)
print(f"resuming with {len(done)} scripts already cached, {frames_done}/{TARGET_FRAMES} frames")
t0 = time.time()

def flush(buf, target_words):
    texts = [b["script"] for b in buf]
    voices = [[b["prompt"]] for b in buf]
    inputs = processor(text=texts, voice_samples=voices, return_tensors="pt", padding=True)
    inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}
    sigma_fn = make_sigma_fn()
    cap_holder = {}
    noise = NoiseIntervention(
        model.model.acoustic_connector, sigma_fn,
        active_fn=lambda: bool(cap_holder) and len(cap_holder["cap"].calls) > 0,
        calls_fn=lambda: len(cap_holder["cap"].calls) if cap_holder else 0,
    )
    cap = BatchedSampleCapture(model, noise=noise)
    cap_holder["cap"] = cap
    try:
        with noise, cap, torch.inference_mode():
            out = model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3,
                                 max_new_tokens=int(target_words * 4))  # generous headroom over
                                                                        # ~2.65 frames/word @ 170wpm/7.5Hz
        seq = out.sequences if hasattr(out, "sequences") else out
        pl = inputs["input_ids"].shape[1]
        streams = [seq[i, pl:].tolist() for i in range(len(buf))]
        parts = cap.split_utterances_v2(streams, frame_id=FRAME_ID)
        n_frames = 0
        for b, (hidden, latent, neg_hidden, sigma) in zip(buf, parts):
            utt = UtteranceCache(
                utt_id=b["uid"], text=b["script"], hidden=hidden, latent=latent,
                meta={"target_words": target_words, "actual_words": b["actual_words"],
                      "batched": True, "capture": "v2"},
                neg_hidden=neg_hidden, sigma=sigma,
            )
            save_utterance(utt, f"{CACHE_DIR}/{b['uid']}.pt")
            done.add(b["uid"])
            n_frames += hidden.shape[0]
        return n_frames
    except Exception as e:
        print("batch failed (skipped, will not retry this session):", repr(e)[:150])  # message
        return 0                                                                       # only --
                                                                                        # never store
                                                                                        # the exception
                                                                                        # object (pins
                                                                                        # GPU memory,
                                                                                        # 2026-08-14
                                                                                        # OOM lesson)

bin_i = 0
buf, sent_pool = [], []
for ex in ds:
    if frames_done >= TARGET_FRAMES:
        break
    text = ex["text_normalized"].strip()
    if not (30 <= len(text) <= 180):
        continue
    sent_pool.append(text)
    target_words = WORD_BINS[bin_i % len(WORD_BINS)]
    if sum(len(s.split()) for s in sent_pool) < target_words:
        continue
    script, actual_words = turnscript(sent_pool, target_words)
    sent_pool = []
    bin_i += 1
    uid = f"cv2_{target_words}w_{uuid.uuid4().hex[:8]}"
    buf.append({"uid": uid, "script": script, "actual_words": actual_words,
               "prompt": random.choice(prompts)})
    bs = BIN_BATCH[target_words]
    if len(buf) == bs:
        n = flush(buf, target_words)
        frames_done += n
        buf = []
        elapsed = time.time() - t0
        print(f"{frames_done}/{TARGET_FRAMES} frames  {len(done)} scripts  {elapsed/3600:.2f}h elapsed")
if buf:  # flush any partial trailing batch
    target_words = WORD_BINS[(bin_i - 1) % len(WORD_BINS)]
    frames_done += flush(buf, target_words)
print(f"DONE: {len(done)} scripts, {frames_done} frames")

In [ ]:
# ===== Summary manifest (sigma histogram + word-bin distribution) -> Drive =====
files = list(Path(CACHE_DIR).glob("*.pt"))
sigmas, words, total_frames = [], [], 0
for f in files:
    d = torch.load(f, weights_only=True)
    total_frames += d["hidden"].shape[0]
    words.append(d["meta"]["target_words"])
    if d.get("sigma") is not None:
        sigmas.extend(float(s) for s in d["sigma"])

def hist(vals, buckets):
    out = {}
    for lo, hi, name in buckets:
        out[name] = sum(1 for v in vals if lo <= v < hi)
    return out

report = {
    "n_scripts": len(files),
    "total_frames": total_frames,
    "target_frames": TARGET_FRAMES,
    "word_bin_counts": {w: words.count(w) for w in sorted(set(words))},
    "sigma_window_histogram": hist(sigmas, [
        (0.0, 0.001, "0"), (0.001, 0.1, "(0,0.1)"), (0.1, 0.3, "[0.1,0.3)"),
        (0.3, 0.35, "0.3"), (0.35, 1.0, "0.4+"),
    ]),
    "n_sigma_windows": len(sigmas),
}
print(json.dumps(report, indent=2))
with open(f"{CACHE_DIR}/capture_v2_manifest.json", "w") as f:
    json.dump(report, f, indent=2)
print(f"manifest written to {CACHE_DIR}/capture_v2_manifest.json")